# Setup

In [1]:
from shapely.geometry import box
import geopandas as gpd
import leafmap
import pandas as pd
import geoai
import torch
import glob
import rasterio

from plotly.subplots import make_subplots
import plotly.graph_objects as go

import numpy as np
from rasterio.features import rasterize



In [2]:
# the Syke peat after-use layer
peat = gpd.read_file("../data/raw/vectors/turvetuotantoalueet/turvetuotantoalueet_jalkikaytto.shp")

# leafmap needs WGS84 for display
peat_wgs = peat.to_crs("EPSG:4326")

# one colour per after-use class
colors = {
    "avoin": "#ff6600",
    "metsittynyt": "#00e64d",
    "turvetuotanto": "#cc00ff",
    "vesi": "#00ccff",
    "maatalous": "#ffee00",
    "rakennettu": "#ff0033",}

# same colours with English labels for the legend
legend = {
    "Open": "#ff6600",
    "Afforested": "#00e64d",
    "Still in production": "#cc00ff",
    "Water": "#00ccff",
    "Agriculture": "#ffee00",
    "Built-up": "#ff0033",}

In [3]:
# English labels for the Finnish class names
class_names_en = {
    "avoin": "open",
    "metsittynyt": "afforested",
    "turvetuotanto": "still_in_production",
    "vesi": "water",
    "maatalous": "agriculture",
    "rakennettu": "built_up",
}

peat_wgs["class_en"] = peat_wgs["luokka"].map(class_names_en)


# Inspection of training data

In [4]:
# slide a 0.4 x 0.3 degree window over the region and count what falls inside
# finding AOI candidates
results = []

for west in [21.8, 22.0, 22.2, 22.4, 22.6, 22.8, 23.0, 23.2, 23.4, 23.6, 23.8]:
    for south in [62.0, 62.2, 62.4, 62.6, 62.8, 63.0]:
        east = west + 0.4
        north = south + 0.3
        window = peat_wgs.cx[west:east, south:north]
        counts = window["class_en"].value_counts()
        results.append({
            "west": west,
            "south": south,
            "still_in_production": counts.get("still_in_production", 0),
            "open": counts.get("open", 0),
            "total": len(window),
        })

candidates = pd.DataFrame(results)
print(candidates.sort_values("still_in_production", ascending=False).head(15))

    west  south  still_in_production  open  total
19  22.4   62.2                  193  1576   2399
43  23.2   62.2                  192  2349   3779
37  23.0   62.2                  176  2558   4229
2   21.8   62.4                  171   696   1155
25  22.6   62.2                  154  1990   2914
24  22.6   62.0                  149  2177   3549
26  22.6   62.4                  139  1548   2331
20  22.4   62.4                  138  1062   1579
44  23.2   62.4                  122  1955   3294
18  22.4   62.0                  119  1075   1893
12  22.2   62.0                  117  1120   1757
50  23.4   62.4                  113  1400   2116
32  22.8   62.4                  112  1440   2249
31  22.8   62.2                  112  1853   2857
13  22.2   62.2                  112   715   1198


In [5]:
from shapely.geometry import box

train01 = box(22.4, 62.2, 22.8, 62.5).buffer(0.1)
train02 = box(21.8, 62.4, 22.2, 62.7).buffer(0.1)
test01 = box(23.2, 62.2, 23.6, 62.5).buffer(0.1)

results = []

for west in [21.4, 21.6, 21.8, 22.0, 22.2, 22.4, 22.6, 22.8, 23.0, 23.2, 23.4, 23.6, 23.8, 24.0]:
    for south in [61.8, 62.0, 62.2, 62.4, 62.6, 62.8, 63.0, 63.2]:
        east = west + 0.4
        north = south + 0.3

        window_geom = box(west, south, east, north)
        if window_geom.intersects(train01) or window_geom.intersects(train02) or window_geom.intersects(test01):
            continue

        window = peat_wgs.cx[west:east, south:north]
        counts = window["class_en"].value_counts()
        results.append({
            "west": west,
            "south": south,
            "still_in_production": counts.get("still_in_production", 0),
            "open": counts.get("open", 0),
            "afforested": counts.get("afforested", 0),
            "total": len(window),
        })

candidates = pd.DataFrame(results)
print(candidates.sort_values("still_in_production", ascending=False).head(15))

    west  south  still_in_production  open  afforested  total
10  22.0   61.8                  180  2468         760   3640
13  22.2   61.8                  153  2598         932   3867
57  24.0   62.8                  119  3209         670   4074
7   21.8   61.8                  112   659         108    996
58  24.0   63.0                   85  2588         564   3325
20  22.6   61.8                   85  1919         805   2937
55  24.0   62.4                   83  1651         404   2169
51  23.8   63.2                   76  1025         196   1352
16  22.4   61.8                   75  1236         493   1890
34  23.2   63.0                   71  1378         234   1743
49  23.8   62.8                   70   334          65    504
4   21.6   61.8                   70   546         155    817
24  22.8   61.8                   67  1398         546   2103
43  23.6   63.2                   65   703         198   1014
48  23.8   62.6                   62   229          41    365


In [ ]:
# training region 01
train_region01_2021 = {"west": 22.4, "south": 62.2, "east": 22.8, "north": 62.5}

# training region 02
train_region02_2021 = {"west": 21.8, "south": 62.4, "east": 22.2, "north": 62.7}

train_region_03_2021 = {"west": 22.0, "south": 62.0, "east": 22.4, "north": 62.3}


train_region_04_2021 = {"west": 23.2, "south": 62.6, "east": 23.6, "north": 62.9}
# test region01, 192 production sites
test_region01_2021 = {"west": 23.2, "south": 62.2, "east": 23.6, "north": 62.5}

train_box_01 = gpd.GeoDataFrame(
    {"name": ["train01"]},
    geometry=[box(train_region01_2021["west"], train_region01_2021["south"], train_region01_2021["east"], train_region01_2021["north"])],
    crs="EPSG:4326",)

train_box_02 = gpd.GeoDataFrame(
    {"name": ["train02"]},
    geometry=[box(train_region02_2021["west"], train_region02_2021["south"], train_region02_2021["east"], train_region02_2021["north"])],
    crs="EPSG:4326",)

train_box_03 = gpd.GeoDataFrame(
    {"name": ["train03"]},
    geometry=[box(train_region03_2021["west"], train_region03_2021["south"], train_region03_2021["east"], train_region03_2021["north"])],
    crs="EPSG:4326",)

train_box_04 = gpd.GeoDataFrame(
    {"name": ["train04"]},
    geometry=[box(train_region_04_2021["west"], train_region_04_2021["south"], train_region_04_2021["east"], train_region_04_2021["north"])],
    crs="EPSG:4326",)

test_box_01 = gpd.GeoDataFrame(
    {"name": ["test01"]},
    geometry=[box(test_region01_2021["west"], test_region01_2021["south"], test_region01_2021["east"], test_region01_2021["north"])],
    crs="EPSG:4326",)

# regions added to the legend
legend_with_boxes = dict(legend)
legend_with_boxes["Train region 01"] = "#ffffff"
legend_with_boxes["Train region 02"] = "#0000ff"
legend_with_boxes["Train region 03"] = "#00ff00"
legend_with_boxes["Train region 04"] = "#ff0000"
legend_with_boxes["Test region"] = "#000000"

# only the polygons near the boxes, drawing all of Finland would hang the browser
peat_near = peat_wgs.cx[21.6:24.0, 62.0:63.3]

m = leafmap.Map(center=[62.5, 22.7], zoom=9)

# The layer "rakennettu" is left out as there is only one polygon in the whole dataset.
# one layer per class, each gets its own toggle in the layer control
m.add_gdf(peat_near[peat_near["luokka"] == "avoin"], layer_name="Open",
          style={"color": "#ff6600", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "metsittynyt"], layer_name="Afforested",
          style={"color": "#00e64d", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "turvetuotanto"], layer_name="Still in production",
          style={"color": "#cc00ff", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "vesi"], layer_name="Water",
          style={"color": "#00ccff", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "maatalous"], layer_name="Agriculture",
          style={"color": "#ffee00", "weight": 2, "fill": False})

# the three regions on top
m.add_gdf(train_box_01, layer_name="Train region 01", style={"color": "#ffffff", "weight": 3, "fill": False})
m.add_gdf(train_box_02, layer_name="Train region 02", style={"color": "#0000ff", "weight": 3, "fill": False})
m.add_gdf(train_box_03, layer_name="Train region 03", style={"color": "#00ff00", "weight": 3, "fill": False})
m.add_gdf(train_box_04, layer_name="Train region 04", style={"color": "#ff0000", "weight": 3, "fill": False})
m.add_gdf(test_box_01, layer_name="Test region 01", style={"color": "#000000", "weight": 3, "fill": False})

m.add_legend(title="After-use class", legend_dict=legend_with_boxes)
m

Map(center=[62.5, 22.7], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [11]:
train_region01 = peat_wgs.cx[22.4:22.8, 62.2:62.5]
train_region02 = peat_wgs.cx[21.8:22.2, 62.4:62.7]
train_region03 = peat_wgs.cx[22.0:22.4, 62.0:62.3]

print("TRAIN_Region01")
print(train_region01["class_en"].value_counts())

print("TRAIN_Region02")
print(train_region02["class_en"].value_counts())

print("TRAIN_Region03")
print(train_region03["class_en"].value_counts())

TRAIN_Region01
class_en
open                   1576
afforested              388
still_in_production     193
agriculture             125
water                   117
Name: count, dtype: int64
TRAIN_Region02
class_en
open                   696
still_in_production    171
afforested             127
agriculture             96
water                   65
Name: count, dtype: int64
TRAIN_Region03
class_en
open                   1026
afforested              266
still_in_production     101
agriculture              77
water                    35
Name: count, dtype: int64


In [12]:
train_region04 = peat_wgs.cx[23.2:23.6, 62.6:62.9]
test_region01 = peat_wgs.cx[23.2:23.6, 62.2:62.5]

print("TRAIN_Region04")
print(train_region04["class_en"].value_counts())

print("TEST_Region01")
print(test_region01["class_en"].value_counts())

TRAIN_Region04
class_en
open                   622
afforested             168
still_in_production     48
agriculture             41
water                   23
Name: count, dtype: int64
TEST_Region01
class_en
open                   2349
afforested             1084
still_in_production     192
agriculture              80
water                    74
Name: count, dtype: int64


In [13]:
# Checking the raster data for the training region01
geoai.print_raster_info("../data/raw/raster/train_region01_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/train_region01_2021.tif =====
Driver: GTiff
Dimensions: 2155 x 3394 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=572110.0, bottom=6897240.0, right=593660.0, top=6931180.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 45.00
  Max: 12708.00
  Mean: 310.46
  Std Dev: 133.98
Band 2:
  Min: 64.00
  Max: 12756.00
  Mean: 480.11
  Std Dev: 181.66
Band 3:
  Min: 49.00
  Max: 10376.00
  Mean: 369.77
  Std Dev: 224.01
Band 4:
  Min: 38.00
  Max: 12124.00
  Mean: 2537.19
  Std Dev: 831.95
Band 5:
  Min: 25.00
  Max: 6211.00
  Mean: 1567.23
  Std Dev: 612.32
Band 6:
  Min: 14.00
  Max: 5990.00
  Mean: 859.62
  Std Dev: 486.57


In [14]:
# Checking the raster data for the training region02
geoai.print_raster_info("../data/raw/raster/train_region02_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/train_region02_2021.tif =====
Driver: GTiff
Dimensions: 2110 x 3375 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=540930.0, bottom=6918990.0, right=562030.0, top=6952740.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 108.00
  Max: 9160.00
  Mean: 336.67
  Std Dev: 157.96
Band 2:
  Min: 89.00
  Max: 8756.00
  Mean: 515.17
  Std Dev: 201.34
Band 3:
  Min: 62.00
  Max: 9064.00
  Mean: 389.56
  Std Dev: 246.56
Band 4:
  Min: 70.00
  Max: 9824.00
  Mean: 2730.69
  Std Dev: 750.01
Band 5:
  Min: 45.00
  Max: 6191.00
  Mean: 1581.82
  Std Dev: 586.15
Band 6:
  Min: 27.00
  Max: 6111.00
  Mean: 863.34
  Std Dev: 469.23


In [15]:
geoai.print_raster_info("../data/raw/raster/train_region03_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/train_region03_2021.tif =====
Driver: GTiff
Dimensions: 2147 x 3381 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=551860.0, bottom=6874580.0, right=573330.0, top=6908390.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 106.00
  Max: 4896.00
  Mean: 306.20
  Std Dev: 115.28
Band 2:
  Min: 76.00
  Max: 4724.00
  Mean: 471.42
  Std Dev: 157.79
Band 3:
  Min: 55.00
  Max: 4804.00
  Mean: 356.21
  Std Dev: 202.43
Band 4:
  Min: 58.00
  Max: 6552.00
  Mean: 2473.67
  Std Dev: 696.62
Band 5:
  Min: 31.00
  Max: 5109.00
  Mean: 1509.13
  Std Dev: 486.76
Band 6:
  Min: 17.00
  Max: 5154.00
  Mean: 809.68
  Std Dev: 385.29


In [16]:
geoai.print_raster_info("../data/raw/raster/train_region04_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/train_region04_2021.tif =====
Driver: GTiff
Dimensions: 2167 x 3418 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=611800.0, bottom=6942940.0, right=633470.0, top=6977120.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 93.00
  Max: 12008.00
  Mean: 320.58
  Std Dev: 135.25
Band 2:
  Min: 66.00
  Max: 12012.00
  Mean: 487.09
  Std Dev: 192.16
Band 3:
  Min: 48.00
  Max: 11100.00
  Mean: 371.93
  Std Dev: 227.84
Band 4:
  Min: 49.00
  Max: 10980.00
  Mean: 2546.48
  Std Dev: 886.67
Band 5:
  Min: 16.00
  Max: 8009.00
  Mean: 1529.95
  Std Dev: 640.53
Band 6:
  Min: 9.00
  Max: 8523.00
  Mean: 817.24
  Std Dev: 448.94


In [9]:
# Checking if production sites look different from open sites
peat_aoi = peat_wgs.cx[22.4:22.8, 62.2:62.5]

m = leafmap.Map(center=[62.35, 22.6], zoom=11)
m.add_raster("../data/raw/raster/train_ndvi_2021.tif", colormap="RdYlGn", vmin=0, vmax=0.9, layer_name="NDVI")

for cls, c in colors.items():
    sub = peat_aoi[peat_aoi["luokka"] == cls]
    if len(sub) > 0:
        m.add_gdf(sub, layer_name=cls, style={"color": c, "weight": 2, "fill": False})

m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.349972, 22.6020605], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [21]:
# Checking polygons by area rather than polygon count
region01 = peat_wgs.cx[22.4:22.8, 62.2:62.5].to_crs(3067)
region01["area_ha"] = region01.area / 10000

region02 = peat_wgs.cx[21.8:22.2, 62.4:62.7].to_crs(3067)
region02["area_ha"] = region02.area / 10000

region03 = peat_wgs.cx[22.0:22.4, 62.0:62.3].to_crs(3067)
region03["area_ha"] = region03.area / 10000

print("TRAIN_Region01 area in hectares:")
print(region01.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

print("TRAIN_Region02 area in hectares:")
print(region02.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

print("TRAIN_Region03 area in hectares:")
print(region03.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))


TRAIN_Region01 area in hectares:
class_en
still_in_production    2288.969268
agriculture            1739.232797
open                    582.307073
afforested               79.734218
water                    16.199157
Name: area_ha, dtype: float64
TRAIN_Region02 area in hectares:
class_en
still_in_production    1847.385590
agriculture             534.275649
open                    155.460795
afforested               25.996707
water                    21.915044
Name: area_ha, dtype: float64
TRAIN_Region03 area in hectares:
class_en
still_in_production    1126.329612
agriculture             381.643354
open                    138.369863
afforested               89.429959
water                    23.025111
Name: area_ha, dtype: float64


In [22]:

region04 = peat_wgs.cx[23.2:23.6, 62.6:62.9].to_crs(3067)
region04["area_ha"] = region04.area / 10000

test01 = peat_wgs.cx[23.2:23.6, 62.2:62.5].to_crs(3067)
test01["area_ha"] = test01.area / 10000

print("TRAIN_Region04 area in hectares:")
print(region04.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

print("TEST_Region01 area in hectares:")
print(test01.groupby("class_en")["area_ha"].sum().sort_values(ascending=False))

TRAIN_Region04 area in hectares:
class_en
still_in_production    935.305669
open                   205.746127
agriculture             64.391336
afforested              17.811965
water                    4.954711
Name: area_ha, dtype: float64
TEST_Region01 area in hectares:
class_en
still_in_production    1303.078343
open                    657.139798
agriculture             425.354479
afforested              117.506111
water                    47.522672
Name: area_ha, dtype: float64


In [12]:
# production polygons clipped to the training region01, in the raster's CRS
train_labels = peat_wgs.cx[22.4:22.8, 62.2:62.5]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/train_labels_binary/train_labels_binary_region01_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

193 polygons, CRS EPSG:32634


In [11]:
# production polygons clipped to the training region02, in the raster's CRS
train_labels = peat_wgs.cx[21.8:22.2, 62.4:62.7]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/train_labels_binary/train_labels_binary_region02_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

171 polygons, CRS EPSG:32634


In [23]:
# production polygons clipped to the training region03, in the raster's CRS
train_labels = peat_wgs.cx[22.0:22.4, 62.0:62.3]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/train_labels_binary/train_labels_binary_region03_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

101 polygons, CRS EPSG:32634


In [24]:
# production polygons clipped to the training region04, in the raster's CRS
train_labels = peat_wgs.cx[23.2:23.6, 62.6:62.9]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/train_labels_binary/train_labels_binary_region04_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

48 polygons, CRS EPSG:32634


In [13]:
# production polygons clipped to the test region01, in the raster's CRS
test_labels = peat_wgs.cx[23.2:23.6, 62.2:62.5]
test_labels = test_labels[test_labels["class_en"] == "still_in_production"]
test_labels = test_labels.to_crs("EPSG:32634")

test_labels.to_file("../data/processed/test_labels_binary_region01_2021.gpkg", driver="GPKG")
print(len(test_labels), "polygons, CRS", test_labels.crs)

192 polygons, CRS EPSG:32634


# Tiling the Imagery

In [25]:
stats = geoai.export_geotiff_tiles_batch(
    images_folder="../data/processed/train_images",
    masks_folder="../data/processed/train_labels_binary",
    output_folder="../data/training/tiles_binary_4regions_clean",
    tile_size=256,
    stride=128,
    class_value_field="class",
    match_by_name=False,
)

print(stats)

Processing image pairs:   0%|          | 0/4 [00:00<?, ?it/s]

{'total_image_pairs': 4, 'processed_pairs': 4, 'total_tiles': 1664, 'tiles_with_features': 636, 'errors': 0, 'processed_files': [{'image': '../data/processed/train_images\\train_region01_2021_clean.tif', 'mask': '../data/processed/train_labels_binary\\train_labels_binary_region01_2021.gpkg', 'base_name': 'train_region01_2021_clean', 'tiles_generated': 416, 'tiles_with_features': 241}, {'image': '../data/processed/train_images\\train_region02_2021_clean.tif', 'mask': '../data/processed/train_labels_binary\\train_labels_binary_region02_2021.gpkg', 'base_name': 'train_region02_2021_clean', 'tiles_generated': 416, 'tiles_with_features': 186}, {'image': '../data/processed/train_images\\train_region03_2021_clean.tif', 'mask': '../data/processed/train_labels_binary\\train_labels_binary_region03_2021.gpkg', 'base_name': 'train_region03_2021_clean', 'tiles_generated': 416, 'tiles_with_features': 107}, {'image': '../data/processed/train_images\\train_region04_2021_clean.tif', 'mask': '../data/pr

In [26]:
# check that tiling worked
images = sorted(glob.glob("../data/training/tiles_binary_4regions_clean/images/*.tif"))
masks = sorted(glob.glob("../data/training/tiles_binary_4regions_clean/masks/*.tif"))
print(len(images), "images,", len(masks), "masks")

maxes = []
for path in masks:
    with rasterio.open(path) as src:
        maxes.append(int(src.read(1).max()))

print("highest value in any mask:", max(maxes))
print("masks above 1:", sum(m > 1 for m in maxes))

1664 images, 1664 masks
highest value in any mask: 1
masks above 1: 0


In [27]:
# how many tiles contain production, and how many positive pixels in total
counts = []
for path in masks:
    with rasterio.open(path) as src:
        counts.append(int(src.read(1).sum()))

print("tiles with something:", sum(c > 0 for c in counts))
print("total positive pixels:", sum(counts))

tiles with something: 636
total positive pixels: 2300913


In [28]:
# look at the tile with the most production pixels
n = counts.index(max(counts))

m = leafmap.Map()
m.add_raster(images[n], bands=[3, 2, 1], vmin=0, vmax=1500, layer_name="Sentinel-2")
m.add_raster(masks[n], colormap="Reds", vmin=0, vmax=1, nodata=0, layer_name="Mask")
m

Map(center=[62.2615895, 22.561329], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

# Runing the binary_model

The model contains 2 training regions which have been cleaned and the lr = 0.0001

In [ ]:
geoai.train_segmentation_model(
    images_dir="../data/training/tiles_binary_4regions_clean/images",
    labels_dir="../data/training/tiles_binary_4regions_clean/masks",
    output_dir="../models/binary_4regions_clean",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=6,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.0001,
    val_split=0.2,
)

# Evaluating the binary Model

In [29]:
history = torch.load("../models/binary_4regions_clean/training_history.pth", weights_only=False)

df = pd.DataFrame({
    "epoch": range(1, len(history["train_losses"]) + 1),
    "train_loss": history["train_losses"],
    "val_loss": history["val_losses"],
    "val_iou": history["val_ious"],
    "val_f1": history["val_f1s"],
    "val_precision": history["val_precisions"],
    "val_recall": history["val_recalls"],
})

print("best epoch by IoU:")
print(df.loc[df["val_iou"].idxmax()])
df

best epoch by IoU:
epoch            20.000000
train_loss        0.062072
val_loss          0.020686
val_iou           0.885609
val_f1            0.907502
val_precision     0.939003
val_recall        0.929104
Name: 19, dtype: float64


,epoch,train_loss,val_loss,val_iou,val_f1,val_precision,val_recall
0,1,0.171934,0.069542,0.707203,0.732810,0.756484,0.891223
1,2,0.094767,0.037452,0.826883,0.850623,0.878682,0.913916
2,3,0.083415,0.029995,0.876912,0.898378,0.952948,0.903646
3,4,0.078732,0.025624,0.877949,0.900274,0.931485,0.923199
4,5,0.076830,0.028487,0.841776,0.866128,0.885081,0.932934
5,6,0.074263,0.032008,0.750369,0.776210,0.776292,0.946480
6,7,0.070917,0.025269,0.847599,0.871059,0.889460,0.936117
7,8,0.072800,0.033924,0.793970,0.818065,0.852067,0.906035
8,9,0.068214,0.024495,0.879223,0.901650,0.948385,0.912163
9,10,0.068935,0.022917,0.843920,0.868108,0.884758,0.936711


In [30]:
fig = make_subplots(rows=1, cols=5, subplot_titles=["Loss", "IoU", "F1", "Precision", "Recall"])

fig.add_trace(go.Scatter(x=df["epoch"], y=df["train_loss"], name="Train Loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_loss"], name="Val Loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_iou"], name="Val IoU"), row=1, col=2)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_f1"], name="Val F1"), row=1, col=3)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_precision"], name="Val Precision"), row=1, col=4)
fig.add_trace(go.Scatter(x=df["epoch"], y=df["val_recall"], name="Val Recall"), row=1, col=5)

fig.update_layout(height=300, width=1000)
fig.write_html("../results/figures/training_curves_binary_4regions_clean.html")
fig.show()

# Run Inference

In [34]:
masks_path = "../results/predictions/prediction_binary_4regions_clean_test_region01_2021.tif"
probability_path = "../results/predictions/probability_binary_4regions_clean_test_region01_2021.tif"
model_path = "../models/binary_4regions_clean/best_model.pth"

geoai.semantic_segmentation(
    input_path="../data/processed/test_region01_2021_clean.tif",
    output_path=masks_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=6,
    num_classes=2,
    window_size=256,
    overlap=128,
    batch_size=4,
    probability_path=probability_path,
)

  0%|          | 0/442 [00:00<?, ?it/s]

# Inspecting the mask

In [35]:
m = leafmap.Map()
m.add_raster("../data/processed/test_region01_2021_clean.tif", bands=[3, 2, 1], vmin=0, vmax=1500, layer_name="Sentinel-2")
m.add_raster(masks_path, colormap="Reds", vmin=0, vmax=1, nodata=0, layer_name="Prediction")
m

Map(center=[62.3499595, 23.402183], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [36]:
m = leafmap.Map()
m.add_raster("../data/processed/test_region01_2021_clean.tif", indexes=[3, 2, 1], vmin=0, vmax=1500, layer_name="Sentinel-2")
m.add_raster(probability_path, indexes=[2], colormap="viridis", vmin=0, vmax=1, layer_name="Probability")
m

Map(center=[62.3499595, 23.402183], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

# Vectorise and Filter

In [37]:
output_vector_path = "../results/predictions/prediction_binary_4regions_clean_test_region01_2021.geojson"

gdf = geoai.orthogonalize(
    masks_path,
    output_vector_path,
    epsilon=1,
)

gdf_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")
gdf_filtered = gdf_props[(gdf_props["area_m2"] > 10000)]


Converting features:   0%|          | 0/229 [00:00<?, ?shape/s]

In [40]:
truth = gpd.read_file("../data/processed/test_labels_binary_region01_2021.gpkg").to_crs("EPSG:4326")
pred_polys = gdf_filtered.to_crs("EPSG:4326")

legend_pred = {
    "Syke production sites": "#00ff00",
    "Predicted production": "#ff0000",
}

m = leafmap.Map()
m.add_raster("../data/processed/test_region01_2021_clean.tif", indexes=[3, 2, 1], vmin=0, vmax=1500, layer_name="Sentinel-2")
m.add_gdf(truth, layer_name="Syke production sites", style={"color": "#00ff00", "weight": 2, "fill": False})
m.add_gdf(pred_polys, layer_name="Predicted", style={"color": "#ff0000", "weight": 2, "fill": False})
m.add_legend(title="Test region 2021", legend_dict=legend_pred)
m

Map(center=[62.3499595, 23.402183], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

# Error map and error quantification

In [43]:
with rasterio.open(masks_path) as src:
    pred = src.read(1)
    profile = src.profile
    truth = rasterize(
        gpd.read_file("../data/processed/test_labels_binary_region01_2021.gpkg").to_crs(src.crs).geometry,
        out_shape=src.shape,
        transform=src.transform,
        fill=0,
        default_value=1,
        dtype="uint8",
    )

true_pos = int(((pred == 1) & (truth == 1)).sum())
false_pos = int(((pred == 1) & (truth == 0)).sum())
false_neg = int(((pred == 0) & (truth == 1)).sum())

precision = true_pos / (true_pos + false_pos)
recall = true_pos / (true_pos + false_neg)
iou = true_pos / (true_pos + false_pos + false_neg)
f1 = 2 * precision * recall / (precision + recall)

print("correct (true positives)", true_pos)
print("false positives", false_pos)
print("missed (false negatives)", false_neg)
print("precision", round(precision, 3))
print("recall", round(recall, 3))
print("iou", round(iou, 3))
print("f1", round(f1, 3))

correct (true positives) 99023
false positives 44844
missed (false negatives) 31019
precision 0.688
recall 0.761
iou 0.566
f1 0.723


In [45]:
# 1 correct, 2 false positive, 3 missed
error = np.zeros(pred.shape, dtype="uint8")
error[(pred == 1) & (truth == 1)] = 1
error[(pred == 1) & (truth == 0)] = 2
error[(pred == 0) & (truth == 1)] = 3

print(np.unique(error, return_counts=True))

(array([0, 1, 2, 3], dtype=uint8), array([7327624,   99023,   44844,   31019]))


In [46]:
# write the error map as a 3 band RGB image so the colours are fixed
rgb_error = np.zeros((3, *error.shape), dtype="uint8")

rgb_error[1][error == 1] = 204                  # correct, green
rgb_error[0][error == 2] = 255                  # false positive, red
rgb_error[2][error == 3] = 255                  # missed, blue

profile.update(count=3, dtype="uint8", nodata=0)
error_path = "../results/maps/errormap_binary_run02_test_region01_2021.tif"

with rasterio.open(error_path, "w", **profile) as dst:
    dst.write(rgb_error)

legend_error = {
    "Correct": "#00cc00",
    "False positive": "#ff0000",
    "Missed": "#0000ff",
}

m = leafmap.Map()
m.add_raster("../data/processed/test_region01_2021_clean.tif", indexes=[3, 2, 1], vmin=0, vmax=1500, layer_name="Sentinel-2")
m.add_raster(error_path, indexes=[1, 2, 3], vmin=0, vmax=255, nodata=0, layer_name="Errors")
m.add_legend(title="Prediction vs Syke", legend_dict=legend_error)
m

Map(center=[62.3499595, 23.402183], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [47]:
# more vertical view of the error map
from rasterio.features import shapes
from shapely.geometry import shape

with rasterio.open(masks_path) as src:
    crs = src.crs
    transform = src.transform

geoms = []
vals = []
for geom, val in shapes(error, mask=error > 0, transform=transform):
    geoms.append(shape(geom))
    vals.append(int(val))

err_gdf = gpd.GeoDataFrame({"type": vals}, geometry=geoms, crs=crs).to_crs("EPSG:4326")
print(len(err_gdf), "polygons")

2587 polygons


In [49]:
# showing error map more vertically
m = leafmap.Map(height="800px", width="700px")
m.add_raster("../data/processed/test_region01_2021_clean.tif", indexes=[3, 2, 1], vmin=0, vmax=1500, layer_name="Sentinel-2")
m.add_gdf(err_gdf[err_gdf["type"] == 2], layer_name="False positive", style={"color": "#ff0000", "weight": 1, "fillOpacity": 0.5})
m.add_gdf(err_gdf[err_gdf["type"] == 3], layer_name="Missed", style={"color": "#0000ff", "weight": 1, "fillOpacity": 0.5})
m.add_gdf(err_gdf[err_gdf["type"] == 1], layer_name="Correct", style={"color": "#00cc00", "weight": 1, "fillOpacity": 0.5})
m.add_legend(title="Prediction vs Syke", legend_dict=legend_error, position="bottomleft")
m

Map(center=[62.3499595, 23.402183], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [50]:
# Save the metrics to a CSV file for later analysis
pd.DataFrame([{
    "model": "binary_4regions_clean",
    "region": "test_region01_2021",
    "tp": true_pos, "fp": false_pos, "fn": false_neg,
    "precision": precision, "recall": recall, "iou": iou, "f1": f1,
}]).to_csv("../results/evaluation/metrics_binary_4regions_clean_test_region01_2021.csv", index=False)